#### SPARK STREAMING


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    BooleanType,
    IntegerType
)

# =========================================================
# 1. Schema
# =========================================================

schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("time_position", LongType(), True),
    StructField("last_contact", LongType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("baro_altitude_m", DoubleType(), True),
    StructField("on_ground", BooleanType(), True),
    StructField("velocity_mps", DoubleType(), True),
    StructField("true_track", DoubleType(), True),
    StructField("vertical_rate_mps", DoubleType(), True),
    StructField("sensors", StringType(), True),
    StructField("geo_altitude_m", DoubleType(), True),
    StructField("squawk", StringType(), True),
    StructField("spi", BooleanType(), True),
    StructField("position_source", IntegerType(), True),
    StructField("category", IntegerType(), True),
    StructField("opensky_timestamp", LongType(), True),
    StructField("ingestion_timestamp", StringType(), True)
])


# =========================================================
# 2. Read JSON files continuously
# =========================================================

df_stream = (
    spark.readStream
    .format("json")
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .load(
        "/Volumes/aeroflux/source/source_data/raw_data/"
    )
)


# =========================================================
# 3. Write stream to Bronze Delta
# =========================================================

query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/aeroflux/bronze_layer/checkpoints/bronze_checkpoints"
    )
    .trigger(availableNow=True)
    .start(
        "/Volumes/aeroflux/bronze_layer/bronze_data"
    )
)